In [1]:
import boto3
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_s3_folder(bucket_name, prefix, local_dir, max_workers=10):
    """
    多线程下载 S3 文件夹到本地

    :param bucket_name: S3 桶名
    :param prefix: S3 前缀（路径）
    :param local_dir: 本地保存目录
    :param max_workers: 最大线程数
    """
    s3 = boto3.client("s3")
    os.makedirs(local_dir, exist_ok=True)

    # 列出所有对象
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" not in page:
            continue
        for obj in page["Contents"]:
            key = obj["Key"]
            if not key.endswith("/"):  # 跳过文件夹占位符
                keys.append(key)

    # 下载任务
    def download_one(key):
        local_path = os.path.join(local_dir, os.path.relpath(key, prefix))
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(bucket_name, key, local_path)
        return key

    # 多线程执行
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_one, key): key for key in keys}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
            try:
                future.result()
            except Exception as e:
                print(f"下载失败: {futures[future]} - {e}")

# 使用
bucket_name = "slot-data-shared-for-ai"
prefix = "raw/data2/export_csv/"
local_dir = "export_csv_files"

download_s3_folder(bucket_name, prefix, local_dir, max_workers=20)


Downloading: 100%|██████████| 78/78 [36:53<00:00, 28.38s/it]   


In [7]:
import pandas as pd
df = pd.read_csv("./export_csv_files/HX1/HX1_202502.csv",sep = '\t')

/tmp/ipykernel_775/3120287644.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./export_csv_files/HX1/HX1_202502.csv",sep = '\t')


In [8]:
import numpy as np
import pandas as pd

def process_bets(df: pd.DataFrame) -> pd.DataFrame:
    # 1. billno 排序准备
    if "billno" in df.columns:
        try:
            df["billno"] = pd.to_numeric(df["billno"], errors="coerce")
        except Exception:
            pass

    # 数值列转为数值
    df["account"] = pd.to_numeric(df["account"], errors="coerce").fillna(0)
    df["cus_account"] = pd.to_numeric(df["cus_account"], errors="coerce").fillna(0)

    # 2. 计算 payout（提前）
    df["payout"] = (df["account"] + df["cus_account"]).round(6)
    df["cny_payout"] = (df["cny_account"] + df["cny_cus_account"]).round(6)

    # 3. 排序：同 gmcode 靠在一起，时间递增
    df = (
        df.sort_values(["gmcode", "billtime", "billno"], ascending=[True, True, True], kind="mergesort")
          .reset_index(drop=True)
    )

    # 4. 标记 game_type
    df["game_type"] = "BG"

    cnt = df.groupby("gmcode", sort=False, observed=False)["gmcode"].transform("size")
    eligible = cnt > 10  
    zero_hit_inclusive = df["payout"].eq(0).groupby(df["gmcode"], sort=False, observed=False).cummax()
    after_first_zero = zero_hit_inclusive.groupby(df["gmcode"], sort=False, observed=False).shift(fill_value=False)
    df.loc[eligible & after_first_zero, "game_type"] = "FG"

    #5. 计算 elimination_num
    df["elimination_num"] = pd.NA

    mask_free = df["game_type"].eq("FG")
    mask_base = ~mask_free

    # 5a. base：同 gmcode 内 1,2,3… 递增
    df.loc[mask_base, "elimination_num"] = (
        df.loc[mask_base].groupby("gmcode", sort=False, observed=False).cumcount() + 1
    )

    # 5b. free：遇到 (account,cus_account) = (0,0) -> 下一条重置为 1
    z = df["account"].eq(0) & df["cus_account"].eq(0) & mask_free
    prev_zero = z.groupby(df["gmcode"], sort=False, observed=False).shift(fill_value=False)
    seg = prev_zero.groupby(df["gmcode"], sort=False, observed=False).cumsum()
    elim_free = (
        df.loc[mask_free]
          .groupby([df.loc[mask_free, "gmcode"], seg[mask_free]], sort=False, observed=False)
          .cumcount() + 1
    )

    df.loc[mask_free, "elimination_num"] = elim_free
    df["elimination_num"] = df["elimination_num"].astype("int64")

    #6. 新增 free_elimination_num
    df["free_elimination_num"] = pd.NA
    zero_flag = df.loc[mask_free, "payout"].eq(0)
    prior_zero_cnt = (
        zero_flag.groupby(df.loc[mask_free, "gmcode"], sort=False, observed=False)
                 .cumsum()
                 .groupby(df.loc[mask_free, "gmcode"], sort=False, observed=False)
                 .shift(fill_value=0)
    )
    df.loc[mask_free, "free_elimination_num"] = (1 + prior_zero_cnt).astype("int64")

    # 7. 新增 multiplier 
    # 规则：base: 1/2/3/≥4 -> 1/2/3/5；free: 1/2/3/≥4 -> 2/4/6/10
    sc4 = df["elimination_num"].clip(upper=4)  # 把 >=4 压到 4 这个桶
    base_map = {1: 1, 2: 2, 3: 3, 4: 5}
    free_map = {1: 2, 2: 4, 3: 6, 4: 10}
    df["multiplier"] = np.where(
        df["game_type"].eq("FG"),
        sc4.map(free_map),
        sc4.map(base_map)
    ).astype("int64")

    # 8. 重新计算 odds（统一用锚定 account 的规则）
    # 1) 为每个 gmcode 找到“elimination_num==1”的首行 account 作为锚定值
    anchor_acc = (
        df.loc[df["elimination_num"].eq(1), ["gmcode", "account"]]
          .drop_duplicates("gmcode", keep="first")
          .set_index("gmcode")["account"]
    )

    # 2) 需要用锚定值的行：payout>0 且 account==0
    use_anchor = df["payout"].gt(0) & df["account"].eq(0)

    # 3) 计算分母 account：命中规则用锚定，否则用本行 account
    denom_account = np.where(use_anchor, df["gmcode"].map(anchor_acc), df["account"])

    # 4) 计算 odds（仅对 payout>0 的行；含除零保护，四舍五入 2 位）
    has_payout = df["payout"].gt(0)
    df["odds"] = np.where(
        has_payout & (denom_account > 0) & (df["multiplier"] > 0),
        (df["payout"] / denom_account / df["multiplier"]).round(2),
        np.nan
    )

    # ========= 9) 新增 round_num =========
    # 从 1 开始，遇到 elimination_num==1 就 +1；否则保持不变
    round_counter = df["elimination_num"].eq(1).cumsum()
    df["round_num"] = round_counter.map(lambda x: f"{x:09d}")
    
    df["free_elimination_num"] = pd.to_numeric(df["free_elimination_num"], errors="coerce").fillna(0).astype("int64")

    df["odds"] = df["odds"].fillna(0)
    return df


In [9]:
output = process_bets(df)

In [10]:
# ========= 9) 导出 & 检查 =========
ordered_cols = [
    "product_id", "gmcode", "billno", "billtime", "loginname",
    "account", "cus_account", "payout", "currency",
    "cny_account", "cny_cus_account", "cny_payout",
    "game_type", "elimination_num", "multiplier",
    "free_elimination_num", "odds", "round_num"
]

# 只保留表里有的列（避免 loginname / product_id 不存在时报错）
final_cols = [c for c in ordered_cols if c in output.columns]

In [11]:
output[final_cols]

,product_id,gmcode,billno,billtime,loginname,account,cus_account,payout,currency,cny_account,cny_cus_account,cny_payout,game_type,elimination_num,multiplier,free_elimination_num,odds,round_num
0,HX1,1862489230521956355,1895107140872345601,2025-02-27 21:42:06,j9huongkyuri,0.0,0.00,0.00,USDT,0.0,0.00,0.00,BG,1,1,0,0.0,000000001
1,HX1,1885665671438760960,1885665671438760960,2025-02-01 20:25:04,j9mading7788,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000000002
2,HX1,1885665680385161216,1885665680385161216,2025-02-01 20:25:06,j9mading7788,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000000003
3,HX1,1885665689855884803,1885665689855884803,2025-02-01 20:25:09,j9mading7788,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000000004
4,HX1,1885665705228003329,1885665705228003329,2025-02-01 20:25:12,j9mading7788,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000000005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10971,HX1,1895326556809669121,1895326556809669121,2025-02-28 12:13:59,j9947884450,0.2,0.04,0.24,USDT,1.4,0.28,1.68,BG,1,1,0,1.2,000007326
10972,HX1,1895326556809669121,1895326571879752192,2025-02-28 12:14:02,j9947884450,0.0,0.00,0.00,USDT,0.0,0.00,0.00,BG,2,2,0,0.0,000007326
10973,HX1,1895326585200856576,1895326585200856576,2025-02-28 12:14:06,j9947884450,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000007327
10974,HX1,1895326595191687168,1895326595191687168,2025-02-28 12:14:08,j9947884450,0.2,-0.20,0.00,USDT,1.4,-1.40,0.00,BG,1,1,0,0.0,000007328


In [5]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd



def process_file(csv_path):
    file_name = os.path.basename(csv_path)
    out_name = os.path.splitext(file_name)[0] + ".parquet"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    try:
        # 处理完全空文件（会抛 EmptyDataError）或有 header 但无数据（返回 empty df with columns）
        try:
            df = pd.read_csv(csv_path,sep = "\t")
        except EmptyDataError:
            # 完全空的 csv（0 bytes），创建一个空 DataFrame
            df = pd.DataFrame()

        # 如果读到的是空 DF（无行），直接保存 empty parquet（保留列名如果有）
        if df.empty:
            # 如果 CSV 有表头但没有数据，df.columns 会保留表头列名
            empty_to_save = df.copy()
            empty_to_save.to_parquet(out_path, index=False)
            print(f"[EMPTY->PARQUET] {csv_path} -> {out_path}")
            return csv_path, True, "empty_csv_created_parquet"

        # 非空则尝试处理
        processed = process_bets(df)
        # processed 也可能为空（但通常不会），直接写出
        processed.to_parquet(out_path, index=False)
        print(f"[OK] {csv_path} -> {out_path}")
        return csv_path, True, None

    except Exception as e:
        print(f"[ERROR] {csv_path} : {e}")
        return csv_path, False, str(e)


def main():
    # 收集所有 csv 文件（包含子目录）
    csv_files = []
    for root, _, files in os.walk(INPUT_DIR):
        for f in files:
            if f.lower().endswith(".csv"):
                csv_files.append(os.path.join(root, f))

    if not csv_files:
        print("没有找到任何 csv 文件。")
        # 仍然创建 success 文件，说明没文件可处理
        success_path = os.path.join(INPUT_DIR, "success")
        with open(success_path, "w") as f:
            f.write("NO_CSV_FOUND\n")
        return

    results = []
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_to_file = {executor.submit(process_file, f): f for f in csv_files}
        for future in as_completed(future_to_file):
            file = future_to_file[future]
            try:
                results.append(future.result())
            except Exception as e:
                results.append((file, False, str(e)))

    # 写 success 文件（记录每个文件的状态）
    success_path = os.path.join(INPUT_DIR, "success")
    with open(success_path, "w") as f:
        for file, ok, err in results:
            if ok:
                f.write(f"SUCCESS: {file} - {err or 'ok'}\n")
            else:
                f.write(f"FAIL: {file} - {err}\n")

    print(f"全部完成，结果写入: {success_path}")



In [4]:
INPUT_DIR = "./export_csv_files/HX1"
OUTPUT_DIR = "./processed_parquet/HX1"
if __name__ == "__main__":
    main()

[ERROR] ./export_csv_files/HX1/HX1_202506.csv : name 'EmptyDataError' is not defined
[ERROR] ./export_csv_files/HX1/HX1_202507.csv : name 'EmptyDataError' is not defined
[ERROR] ./export_csv_files/HX1/HX1_202508.csv : name 'EmptyDataError' is not defined
[OK] ./export_csv_files/HX1/HX1_202505.csv -> ./processed_parquet/HX1/HX1_202505.parquet


/tmp/ipykernel_543/3115356950.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/HX1/HX1_202503.csv -> ./processed_parquet/HX1/HX1_202503.parquet
[OK] ./export_csv_files/HX1/HX1_202502.csv -> ./processed_parquet/HX1/HX1_202502.parquet
[OK] ./export_csv_files/HX1/HX1_202501.csv -> ./processed_parquet/HX1/HX1_202501.parquet
[OK] ./export_csv_files/HX1/HX1_202411.csv -> ./processed_parquet/HX1/HX1_202411.parquet
[OK] ./export_csv_files/HX1/HX1_202504.csv -> ./processed_parquet/HX1/HX1_202504.parquet
[OK] ./export_csv_files/HX1/HX1_202412.csv -> ./processed_parquet/HX1/HX1_202412.parquet
[OK] ./export_csv_files/HX1/HX1_202410.csv -> ./processed_parquet/HX1/HX1_202410.parquet
[OK] ./export_csv_files/HX1/HX1_202409.csv -> ./processed_parquet/HX1/HX1_202409.parquet
[OK] ./export_csv_files/HX1/HX1_202408.csv -> ./processed_parquet/HX1/HX1_202408.parquet
全部完成，结果写入: ./export_csv_files/HX1/success


In [5]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
INPUT_DIR = "./export_csv_files/A01"
OUTPUT_DIR = "./processed_parquet/A01"
if __name__ == "__main__":
    main()

[OK] ./export_csv_files/A01/A01_202408.csv -> ./processed_parquet/A01/A01_202408.parquet
[OK] ./export_csv_files/A01/A01_202409.csv -> ./processed_parquet/A01/A01_202409.parquet
[OK] ./export_csv_files/A01/A01_202411.csv -> ./processed_parquet/A01/A01_202411.parquet
[OK] ./export_csv_files/A01/A01_202410.csv -> ./processed_parquet/A01/A01_202410.parquet


/tmp/ipykernel_543/3115356950.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/A01/A01_202412.csv -> ./processed_parquet/A01/A01_202412.parquet
[OK] ./export_csv_files/A01/A01_202502.csv -> ./processed_parquet/A01/A01_202502.parquet
[OK] ./export_csv_files/A01/A01_202501.csv -> ./processed_parquet/A01/A01_202501.parquet
[OK] ./export_csv_files/A01/A01_202504.csv -> ./processed_parquet/A01/A01_202504.parquet
[OK] ./export_csv_files/A01/A01_202503.csv -> ./processed_parquet/A01/A01_202503.parquet
[OK] ./export_csv_files/A01/A01_202506.csv -> ./processed_parquet/A01/A01_202506.parquet
[OK] ./export_csv_files/A01/A01_202507.csv -> ./processed_parquet/A01/A01_202507.parquet
[OK] ./export_csv_files/A01/A01_202508.csv -> ./processed_parquet/A01/A01_202508.parquet
[OK] ./export_csv_files/A01/A01_202505.csv -> ./processed_parquet/A01/A01_202505.parquet
全部完成，结果写入: ./export_csv_files/A01/success


In [6]:
INPUT_DIR = "./export_csv_files/A02"
OUTPUT_DIR = "./processed_parquet/A02"
if __name__ == "__main__":
    main()

[OK] ./export_csv_files/A02/A02_202408.csv -> ./processed_parquet/A02/A02_202408.parquet
[OK] ./export_csv_files/A02/A02_202409.csv -> ./processed_parquet/A02/A02_202409.parquet
[OK] ./export_csv_files/A02/A02_202410.csv -> ./processed_parquet/A02/A02_202410.parquet


/tmp/ipykernel_543/3115356950.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/A02/A02_202411.csv -> ./processed_parquet/A02/A02_202411.parquet
[OK] ./export_csv_files/A02/A02_202502.csv -> ./processed_parquet/A02/A02_202502.parquet
[OK] ./export_csv_files/A02/A02_202412.csv -> ./processed_parquet/A02/A02_202412.parquet
[OK] ./export_csv_files/A02/A02_202501.csv -> ./processed_parquet/A02/A02_202501.parquet
[OK] ./export_csv_files/A02/A02_202505.csv -> ./processed_parquet/A02/A02_202505.parquet
[OK] ./export_csv_files/A02/A02_202508.csv -> ./processed_parquet/A02/A02_202508.parquet
[OK] ./export_csv_files/A02/A02_202504.csv -> ./processed_parquet/A02/A02_202504.parquet
[OK] ./export_csv_files/A02/A02_202503.csv -> ./processed_parquet/A02/A02_202503.parquet
[OK] ./export_csv_files/A02/A02_202507.csv -> ./processed_parquet/A02/A02_202507.parquet
[OK] ./export_csv_files/A02/A02_202506.csv -> ./processed_parquet/A02/A02_202506.parquet
全部完成，结果写入: ./export_csv_files/A02/success


In [6]:
INPUT_DIR = "./export_csv_files/A04"
OUTPUT_DIR = "./processed_parquet/A04"
if __name__ == "__main__":
    main()

/tmp/ipykernel_775/3694169489.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/A04/A04_202502.csv -> ./processed_parquet/A04/A04_202502.parquet
[OK] ./export_csv_files/A04/A04_202506.csv -> ./processed_parquet/A04/A04_202506.parquet
[OK] ./export_csv_files/A04/A04_202508.csv -> ./processed_parquet/A04/A04_202508.parquet
[OK] ./export_csv_files/A04/A04_202411.csv -> ./processed_parquet/A04/A04_202411.parquet
[OK] ./export_csv_files/A04/A04_202409.csv -> ./processed_parquet/A04/A04_202409.parquet
[OK] ./export_csv_files/A04/A04_202507.csv -> ./processed_parquet/A04/A04_202507.parquet
[OK] ./export_csv_files/A04/A04_202408.csv -> ./processed_parquet/A04/A04_202408.parquet
[OK] ./export_csv_files/A04/A04_202501.csv -> ./processed_parquet/A04/A04_202501.parquet
[OK] ./export_csv_files/A04/A04_202503.csv -> ./processed_parquet/A04/A04_202503.parquet
[OK] ./export_csv_files/A04/A04_202504.csv -> ./processed_parquet/A04/A04_202504.parquet
[OK] ./export_csv_files/A04/A04_202410.csv -> ./processed_parquet/A04/A04_202410.parquet
[OK] ./export_csv_fil

In [3]:
INPUT_DIR = "./export_csv_files/C08"
OUTPUT_DIR = "./processed_parquet/C08"
if __name__ == "__main__":
    main()

/tmp/ipykernel_22927/3115356950.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/C08/C08_202502.csv -> ./processed_parquet/C08/C08_202502.parquet
[OK] ./export_csv_files/C08/C08_202504.csv -> ./processed_parquet/C08/C08_202504.parquet
[OK] ./export_csv_files/C08/C08_202505.csv -> ./processed_parquet/C08/C08_202505.parquet
[OK] ./export_csv_files/C08/C08_202408.csv -> ./processed_parquet/C08/C08_202408.parquet
[OK] ./export_csv_files/C08/C08_202409.csv -> ./processed_parquet/C08/C08_202409.parquet
[OK] ./export_csv_files/C08/C08_202506.csv -> ./processed_parquet/C08/C08_202506.parquet
[OK] ./export_csv_files/C08/C08_202503.csv -> ./processed_parquet/C08/C08_202503.parquet
[OK] ./export_csv_files/C08/C08_202410.csv -> ./processed_parquet/C08/C08_202410.parquet
[OK] ./export_csv_files/C08/C08_202507.csv -> ./processed_parquet/C08/C08_202507.parquet
[OK] ./export_csv_files/C08/C08_202411.csv -> ./processed_parquet/C08/C08_202411.parquet
[OK] ./export_csv_files/C08/C08_202412.csv -> ./processed_parquet/C08/C08_202412.parquet
[OK] ./export_csv_fil

In [4]:
INPUT_DIR = "./export_csv_files/H86"
OUTPUT_DIR = "./processed_parquet/H86"
if __name__ == "__main__":
    main()

[OK] ./export_csv_files/H86/H86_202408.csv -> ./processed_parquet/H86/H86_202408.parquet
[OK] ./export_csv_files/H86/H86_202409.csv -> ./processed_parquet/H86/H86_202409.parquet
[OK] ./export_csv_files/H86/H86_202411.csv -> ./processed_parquet/H86/H86_202411.parquet
[OK] ./export_csv_files/H86/H86_202410.csv -> ./processed_parquet/H86/H86_202410.parquet
[OK] ./export_csv_files/H86/H86_202412.csv -> ./processed_parquet/H86/H86_202412.parquet


/tmp/ipykernel_22927/3115356950.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path,sep = "\t")


[OK] ./export_csv_files/H86/H86_202502.csv -> ./processed_parquet/H86/H86_202502.parquet
[OK] ./export_csv_files/H86/H86_202501.csv -> ./processed_parquet/H86/H86_202501.parquet
[OK] ./export_csv_files/H86/H86_202505.csv -> ./processed_parquet/H86/H86_202505.parquet
[OK] ./export_csv_files/H86/H86_202503.csv -> ./processed_parquet/H86/H86_202503.parquet
[OK] ./export_csv_files/H86/H86_202508.csv -> ./processed_parquet/H86/H86_202508.parquet
[OK] ./export_csv_files/H86/H86_202504.csv -> ./processed_parquet/H86/H86_202504.parquet
[OK] ./export_csv_files/H86/H86_202507.csv -> ./processed_parquet/H86/H86_202507.parquet
[OK] ./export_csv_files/H86/H86_202506.csv -> ./processed_parquet/H86/H86_202506.parquet
全部完成，结果写入: ./export_csv_files/H86/success
